# 0. Background

This is my implementation of the method from [Zhang et al 2004. *Estimation of saturated pixel values in digital color imaging*](zotero://select/items/1_HT3U4PF3). It is a Bayesian approach to estimating the true value of a saturated pixel, based on information in the unsaturated bands.


In [ ]:
from pathlib import Path
import pystac
import rioxarray as rxr
import numpy as np
import rasterio
from rasterio.enums import Resampling

from tqdm.notebook import tqdm

from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window



In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
from helpers.wv2 import load_wv2
from helpers.shared import assert_grid_match

# 1. Config

In [ ]:
from config import *

# -- Catalog ---------------------------------------------------------------------------------
CATALOG                     = load_catalog()
INPUT_ASSET                 = "image"

# -- Output Options --------------------------------------------------------------------------
OUT_DIR                     = Path('../out/reproj/')
MKDIR                       = True      # Make OUT_DIR if it doesn't exist
OUT_NAME                    = "reproj"
OUT_EXT                     = "tif"

# -- Execution Flags -------------------------------------------------------------------------
OVERWRITE                   = False      # Whether the output should be overwritten

In [ ]:
wv2_col = CATALOG.get_child("wv2-imagery")
sd8_col = CATALOG.get_child("sd8-imagery")
items   = list(wv2_col.get_items()) + list(sd8_col.get_items())

tqdm.write(f"{len(items)} item(s) found:")
for item in items:
    tqdm.write(f"ID: {item.id} | Date: {item.datetime} | Path: {item.assets[INPUT_ASSET].href}")

In [ ]:
OUT_DIR.mkdir(exist_ok=MKDIR)

In [ ]:
def reproj(img, out_path, ref, chunks, profile_options, scale_factor = 1e-4):
    with rasterio.open(img) as src:

        src_nodata = src.nodata if src.nodata is not None else 0
        
        with WarpedVRT(
            src,
            crs = ref.rio.crs,
            transform = ref.rio.transform(),
            width = ref.rio.width,
            height = ref.rio.height,
            resampling = Resampling.average,
            nodata = src_nodata
        ) as vrt:
            profile = vrt.profile.copy()
            profile.update(profile_options)
            dtype = profile.get("dtype")

            with rasterio.open(out_path, "w", **profile) as dst:
                row_offsets = range(0, vrt.height, chunks)
                col_offsets = range(0, vrt.width, chunks)
                n_chunks = len(row_offsets) * len(col_offsets)
                pbar_chunks = tqdm(
                    total=n_chunks,
                    desc = "Reprojecting chunks",
                    unit = "chunks",
                    leave = True,
                    dynamic_ncols = True)
                with pbar_chunks:
                    for row_off in row_offsets:
                        for col_off in col_offsets:
                            h = min(chunks, vrt.height - row_off)
                            w = min(chunks, vrt.width - col_off)
                            window = Window(col_off, row_off, w, h)
                            data = vrt.read(window=window).astype(dtype)
                            mask = data == src_nodata
                            data *= scale_factor
                            data[mask] = np.nan
                            dst.write(data, window=window)
                            pbar_chunks.update(1)

In [ ]:
ref_item = max(items, key = lambda item: item.properties["gsd"]) # Coarsest
tgt_items = [item for item in items if item.id != ref_item.id]

# Metadata printout
ref = load_wv2(ref_item.assets[INPUT_ASSET].href, chunks=CHUNKS)
tqdm.write(f"Reference (coarsest): {ref_item.id} | gsd: {ref_item.properties['gsd']}m | CRS: {ref.rio.crs}")
for tgt_item in tgt_items:
    tgt = load_wv2(tgt_item.assets[INPUT_ASSET].href, chunks = CHUNKS)
    tqdm.write(f"Target: {tgt_item.id} | gsd: {tgt_item.properties['gsd']}m | CRS: {tgt.rio.crs}")

In [ ]:
pbar = tqdm(items, desc = "Reprojecting", unit = "scene", dynamic_ncols = True)

for i, item in enumerate(pbar, start = 1):
    pbar.set_description(f"[{i}/{len(items)}] Reprojecting {item.id}")

    out_path = OUT_DIR / f"{item.id}-{OUT_NAME}.{OUT_EXT}"

    # Update catalog
    asset = pystac.Asset(
        href=str(out_path.resolve()),
        media_type=pystac.MediaType.GEOTIFF,
        roles=["data"],
        title = "Reprojected/Resampled Scene (3m)",
    )

    item.add_asset(OUT_NAME, asset)

    # Check if overwriting
    if out_path.exists() and not OVERWRITE:
        tqdm.write(f"Skipping (exists): {out_path.name}")
        continue

    # Reproject
    reproj(
        img = item.assets[INPUT_ASSET].href,
        out_path = out_path,
        ref = ref,
        chunks = CHUNKS,
        profile_options = PROFILE_FLOAT32,
        scale_factor = 1e-4
    )

    # Check grid match
    assert_grid_match(ref_item.assets[INPUT_ASSET].href, out_path)

    # Build Overviews
    with rasterio.open(out_path, "r+") as dst:
        pbar_ovr = tqdm(total=1, leave=True, desc="Building overviews", dynamic_ncols = True)
        with pbar_ovr:
            dst.build_overviews([2, 4, 8, 16, 32], Resampling.average)
            dst.update_tags(ns="rio_overview", resampling="average")
            pbar_ovr.update(1)

    tqdm.write(f"Saved: {out_path.name}")

CATALOG.normalize_hrefs(str(STAC_DIR))
CATALOG.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
tqdm.write("Complete")